# 03 — Analyze, verify and communicate

Run after 02 in a fresh kernel. Python has already written the fact and analytical tables. This notebook independently rereads the source, executes SQL and writes all charts and the analysis brief under power_bi.

In [ ]:
from pathlib import Path
import json
import sys
from decimal import Decimal, ROUND_HALF_UP
from datetime import datetime, timezone
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Image

ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "PROBLEM_STATEMENT.md").is_file())
RAW = ROOT / "data/raw/online_retail_II.csv"
PROCESSED = ROOT / "data/processed"
QUALITY = ROOT / "data/quality"
TABLES = ROOT / "power_bi/tables"
SOURCE_COLUMNS = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "CustomerID", "Country"]
FACT_COLUMNS = ["SourceRow", *SOURCE_COLUMNS, "InvoiceDay", "RevenuePence", "TotalAmount", "Year", "Month", "MonthName", "DayOfWeek", "WeekdayNumber", "Hour"]
sys.path.insert(0, str(ROOT / "sql"))
from run_sql import read_fact, kpis, load_tables, queries, sqlite_results, compare_sql


## 1. Read the generated fact and tables

The SQL helper preserves identifier text when reading CSVs. Inspect its query execution and comparison functions in sql/run_sql.py; the SQL itself is in sql/02_analysis.sql.

In [ ]:
fact = read_fact()
tables = load_tables()
quality = json.loads((QUALITY / "cleaning_summary.json").read_text(encoding="utf-8"))
SOURCE = ROOT / "data/raw" / quality["source_file"]
display(pd.Series(kpis(fact), name="Measured value").to_frame())

## 2. Independently recompute raw-source controls

This pass uses Decimal ROUND_HALF_UP, rather than the cleaning notebook's integer-mill arithmetic. It applies the same stated population rules without calling the cleaning function.

In [ ]:
def write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )

def load_raw(path: Path = RAW) -> pd.DataFrame:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Source not found: {path}. See README.md, Reproduce the analysis."
        )
    if path.suffix.lower() == ".xlsx":
        # UCI's workbook has two annual sheets; reading only the first loses a year.
        frame = pd.concat(
            pd.read_excel(path, sheet_name=None, dtype="string").values(),
            ignore_index=True,
        )
    else:
        frame = pd.read_csv(path, dtype="string")
    frame.columns = frame.columns.str.strip().str.replace(" ", "", regex=False)
    frame = frame.rename(columns={"InvoiceNo": "Invoice", "UnitPrice": "Price"})
    missing = set(SOURCE_COLUMNS) - set(frame.columns)
    if missing or frame.columns.duplicated().any():
        raise ValueError(
            f"Invalid source columns; missing={sorted(missing)} or duplicate headers."
        )
    return frame[SOURCE_COLUMNS].copy()

def source_controls(source: Path) -> dict:
    """Re-read source; compute money with Decimal rather than pipeline mill arithmetic."""
    raw = load_raw(source)
    for col in SOURCE_COLUMNS:
        raw[col] = raw[col].str.strip().replace("", pd.NA)
    customer = pd.to_numeric(raw.CustomerID, errors="coerce")
    qty = pd.to_numeric(raw.Quantity, errors="coerce")
    price = pd.to_numeric(raw.Price, errors="coerce")
    eligible = raw.loc[
        customer.gt(0)
        & customer.mod(1).eq(0)
        & ~raw.Invoice.str.upper().str.startswith("C").fillna(False)
        & raw[["Invoice", "StockCode", "Country"]].notna().all(axis=1)
        & pd.to_datetime(raw.InvoiceDate, errors="coerce", format="mixed").notna()
        & qty.gt(0)
        & qty.mod(1).eq(0)
        & qty.le(2**31 - 1)
        & price.gt(0)
        & price.le(999999999)
        & ((price * 1000) - (price * 1000).round()).abs().le(0.000001)
    ]
    eligible = eligible.drop_duplicates(SOURCE_COLUMNS)
    revenue = sum(
        int(
            (Decimal(str(p)) * Decimal(str(q))).quantize(
                Decimal(".01"), rounding=ROUND_HALF_UP
            )
            * 100
        )
        for p, q in zip(eligible.Price, eligible.Quantity)
    )
    return {
        "sales_lines": len(eligible),
        "revenue_pence": revenue,
        "orders": eligible.Invoice.nunique(),
        "customers": pd.to_numeric(eligible.CustomerID).nunique(),
        "quantity": int(pd.to_numeric(eligible.Quantity).sum()),
    }

## 3. Compare source, fact and real SQL results

The SQLite engine independently computes all nine queries, including each customer's RFM scores and segment. Revenue must reconcile in pence across all analytical tables; calendar, customer keys and cohort bounds are checked as well.

In [ ]:
def validate(
    fact,
    tables,
    source: Path = RAW,
    quality_dir: Path = QUALITY,
    db_path: Path | None = None,
) -> dict:
    metrics = kpis(fact)
    controls = source_controls(source)
    checks = []
    for key, expected in controls.items():
        if metrics[key] != expected:
            raise AssertionError(f"Raw-source reconciliation failed: {key}")
        checks.append({"check": f"raw_source_{key}", "passed": True})
    if (
        fact.SourceRow.duplicated().any()
        or fact.CustomerID.isna().any()
        or fact.RevenuePence.lt(0).any()
    ):
        raise AssertionError("Fact key/null/money invariant failed")
    for name in [
        "orders",
        "customer_metrics",
        "monthly_revenue",
        "country_revenue",
        "product_performance",
        "daily_revenue",
        "hourly_revenue",
        "rfm_segments",
    ]:
        col = "MonetaryPence" if name == "customer_metrics" else "RevenuePence"
        if int(tables[name][col].sum()) != metrics["revenue_pence"]:
            raise AssertionError(f"Revenue does not reconcile: {name}")
        checks.append({"check": f"table_total_{name}", "passed": True})
    customers, cohorts, calendar = (
        tables["customer_metrics"],
        tables["cohort_retention"],
        tables["calendar"],
    )
    if not (customers.CustomerID.is_unique and len(customers) == metrics["customers"]):
        raise AssertionError("Customer key validation failed")
    if not (
        cohorts.RetentionRate.between(0, 1).all()
        and cohorts.loc[cohorts.MonthIndex.eq(0), "RetentionRate"].eq(1).all()
    ):
        raise AssertionError("Cohort bounds validation failed")
    if not (
        calendar.Date.is_unique
        and calendar.Date.diff().dropna().eq(pd.Timedelta(days=1)).all()
    ):
        raise AssertionError("Calendar continuity validation failed")
    checks += [
        {"check": name, "passed": True}
        for name in [
            "fact_invariants",
            "customer_key",
            "cohort_bounds",
            "calendar_continuity",
        ]
    ]
    results = sqlite_results(fact, db_path)
    checks += compare_sql(results, fact, tables)
    evidence = {
        "status": "passed",
        "checks_passed": len(checks),
        "checks": checks,
        "verified_at_utc": datetime.now(timezone.utc).isoformat(),
        "sql_engine": "SQLite",
        "source_controls": controls,
        "power_bi": "Separate native report verification: README.md, Metric rules and validation",
    }
    write_json(quality_dir / "validation_results.json", evidence)
    print(f"Passed {len(checks)} raw-source, invariant and SQL reconciliation groups.")
    return evidence

In [ ]:
evidence = validate(fact, tables, SOURCE)
display(pd.DataFrame(evidence["checks"]))
assert evidence["status"] == "passed"

## 4. Read the findings at the correct grain

Compare months only when complete; compare products by StockCode. A customer may appear in several countries. Retention uses a fixed original cohort denominator, not the current active-customer count.

In [ ]:
display(tables["monthly_revenue"])
display(tables["country_revenue"].head(10))
display(tables["product_performance"].head(10))
display(tables["rfm_segments"])
display(tables["cohort_retention"].head(15))

## 5. Draw the static charts

These figures are local analysis outputs, separate from interactive Power BI visuals. Amber identifies partial months. The recorded hour has no specified timezone.

In [ ]:
FIGURES = ROOT / "power_bi/figures"
MONEY = FuncFormatter(lambda value, _: f"{value / 1_000_000:.1f}m" if abs(value) >= 1_000_000 else f"{value / 1000:.0f}k")

def generate(fact=None, tables=None, output: Path = FIGURES) -> list[Path]:
    fact = read_fact() if fact is None else fact
    tables = load_tables() if tables is None else tables
    output.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.titleweight": "bold",
            "figure.facecolor": "#fbfaf7",
            "axes.facecolor": "#fbfaf7",
        }
    )
    paths = []

    def save(fig, name):
        fig.tight_layout(pad=2)
        path = output / f"{name}.png"
        fig.savefig(path, dpi=160, facecolor=fig.get_facecolor())
        plt.close(fig)
        paths.append(path)

    monthly = tables["monthly_revenue"]
    fig, ax = plt.subplots(figsize=(13, 5))
    labels = pd.to_datetime(monthly.MonthStart).dt.strftime("%Y-%m")
    ax.bar(
        labels,
        monthly.Revenue,
        color=["#24665c" if full else "#c58539" for full in monthly.FullCalendarMonth],
    )
    ax.set(title="Observed monthly sales | GBP", ylabel="Sales (GBP)")
    ax.yaxis.set_major_formatter(MONEY)
    ax.tick_params(axis="x", labelrotation=60)
    fig.text(
        0.04,
        0.01,
        "Amber = partial calendar month. December 2011 ends on 9 December; no full-month growth claim.",
        fontsize=9,
    )
    save(fig, "monthly_revenue")

    daily = tables["daily_revenue"].set_index("WeekdayNumber").reindex(range(1, 8))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(
        ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
        daily.Revenue.fillna(0),
        color="#24665c",
    )
    ax.set(title="Sales by recorded weekday | GBP", ylabel="Sales (GBP)")
    ax.yaxis.set_major_formatter(MONEY)
    save(fig, "daily_revenue")

    hourly = tables["hourly_revenue"].set_index("Hour").reindex(range(24), fill_value=0)
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(hourly.index, hourly.Revenue, marker="o", color="#24665c")
    ax.set(
        title="Sales by recorded invoice hour | GBP",
        xlabel="Source hour (timezone unspecified)",
        ylabel="Sales (GBP)",
        xticks=range(0, 24, 2),
    )
    ax.yaxis.set_major_formatter(MONEY)
    save(fig, "hourly_revenue")

    for name, table, label, title in [
        (
            "country_analysis",
            tables["country_revenue"].head(10),
            "Country",
            "Top countries by sales | GBP",
        ),
        (
            "top_products",
            tables["product_performance"].head(10),
            "Description",
            "Top stock codes by sales | GBP",
        ),
    ]:
        fig, ax = plt.subplots(figsize=(12, 6))
        table = table.iloc[::-1]
        names = table[label].str.slice(0, 42)
        if name == "top_products":
            names = table.StockCode.astype(str) + " | " + names
        ax.barh(names, table.Revenue, color="#24665c")
        ax.set(title=title, xlabel="Sales (GBP)")
        ax.xaxis.set_major_formatter(MONEY)
        save(fig, name)

    for name, label in [
        ("rfm_analysis", "Segment"),
        ("customer_segments", "FrequencySegment"),
    ]:
        summary = (
            tables["customer_metrics"]
            .groupby(label)
            .agg(Customers=("CustomerID", "size"), Revenue=("Monetary", "sum"))
            .sort_values("Revenue")
        )
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
        axes[0].barh(summary.index, summary.Customers, color="#24665c")
        axes[1].barh(summary.index, summary.Revenue, color="#b37939")
        axes[0].set(title="Customer counts", xlabel="Customers")
        axes[1].set(title="Historical sales", xlabel="Sales (GBP)")
        axes[1].xaxis.set_major_formatter(MONEY)
        fig.suptitle(
            "Full-history customer segments | fixed at dataset cutoff",
            fontweight="bold",
        )
        save(fig, name)

    cohort = tables["cohort_retention"].pivot(
        index="CohortMonth", columns="MonthIndex", values="RetentionRate"
    )
    fig, ax = plt.subplots(figsize=(13, 8))
    heat = ax.imshow(cohort, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
    ax.set(
        title="Observed monthly customer retention | partial last month",
        xlabel="Months since first observed purchase",
        ylabel="First observed purchase month",
        yticks=range(len(cohort)),
        yticklabels=cohort.index,
        xticks=range(0, len(cohort.columns), 2),
    )
    fig.colorbar(heat, ax=ax, label="Active customers / original cohort size")
    save(fig, "cohort_retention")

    metrics = kpis(fact)
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.axis("off")
    cards = [
        ("Sales, GBP", f"{metrics['revenue_gbp']:,.2f}"),
        ("Orders", f"{metrics['orders']:,}"),
        ("Customers", f"{metrics['customers']:,}"),
        ("AOV, GBP", f"{metrics['average_order_value_gbp']:,.2f}"),
    ]
    for i, (label, value) in enumerate(cards):
        ax.text(0.03 + i * 0.25, 0.65, label, color="#55615d", transform=ax.transAxes)
        ax.text(
            0.03 + i * 0.25,
            0.35,
            value,
            fontsize=21,
            weight="bold",
            color="#174c44",
            transform=ax.transAxes,
        )
    ax.set_title(
        f"Eligible identified-customer sales | {metrics['first_date']} to {metrics['last_date']}",
        loc="left",
    )
    save(fig, "kpi_summary")
    print(f"Generated {len(paths)} figures in {output}.")
    return paths

In [ ]:
paths = generate(fact, tables)
for path in paths:
    display(Image(filename=str(path)))

## 6. Write a measured business brief

The brief calculates findings from this run. Recommendations remain proposals requiring margin, inventory or campaign evidence; the dataset cannot establish realized savings or causal effects.

In [ ]:
def write_report(fact, tables, quality, evidence):
    m = kpis(fact)
    country = tables["country_revenue"].iloc[0]
    product = tables["product_performance"].iloc[0]
    month = (
        tables["monthly_revenue"]
        .query("FullCalendarMonth")
        .sort_values("RevenuePence", ascending=False)
    )
    peak_month = month.iloc[0] if not month.empty else None
    day = tables["daily_revenue"].sort_values("RevenuePence", ascending=False).iloc[0]
    hour = tables["hourly_revenue"].sort_values("RevenuePence", ascending=False).iloc[0]
    top_count = max(1, round(m["customers"] * 0.2))
    top_share = (
        tables["customer_metrics"]
        .nlargest(top_count, "MonetaryPence")
        .MonetaryPence.sum()
        / m["revenue_pence"]
    )
    excluded = "\n".join(
        f"| {key} | {value:,} |" for key, value in quality["exclusions"].items()
    )
    segment_rows = "\n".join(
        f"| {r.Segment} | {r.Customers:,} | {r.Revenue:,.2f} | {r.RevenueShare:.1%} |"
        for r in tables["rfm_segments"].itertuples()
    )
    text = f"""# Retail sales analysis

Generated from source SHA-256 `{quality["source_sha256"]}`. This is an observed
identified-customer, positive-sales analysis, with cancellations and repeated
source rows excluded. It is not net revenue, profit or achieved business impact.

## Measured baseline

| Measure | Value |
|---|---:|
| Source rows | {quality["source_rows"]:,} |
| Eligible sales lines | {m["sales_lines"]:,} |
| Sales, GBP | {m["revenue_gbp"]:,.2f} |
| Orders | {m["orders"]:,} |
| Customers | {m["customers"]:,} |
| Stock codes | {m["products"]:,} |
| Countries | {m["countries"]:,} |
| Average order value, GBP | {m["average_order_value_gbp"]:,.2f} |
| Repeat customer rate, observed history | {m["repeat_customer_rate"]:.2%} |
| Observed dates | {m["first_date"]} to {m["last_date"]} |

## Findings and proposed actions

1. **Geography:** {country.Country} contributes {country.RevenueShare:.2%} of eligible sales.
   Review domestic inventory exposure and service demand before changing allocation.
2. **Products:** stock code {product.StockCode}, {product.Description}, leads at
   GBP {product.Revenue:,.2f}. Review availability and margin before promoting it;
   cost, stockouts and product categories are absent from the source.
3. **Timing:** the highest complete month is {str(peak_month.MonthStart)[:7] if peak_month is not None else "unavailable"}.
   The leading recorded weekday is {day.DayOfWeek}, and hour {int(hour.Hour):02d}:00
   has the most sales. These are aggregate invoice patterns, not staffing forecasts.
4. **Concentration:** the top {top_count:,} customers (approximately 20%) account for
   {top_share:.2%} of sales. Investigate account concentration and retention using
   the static RFM table; no campaign uplift has been measured.

## RFM at the historical cutoff

| Segment | Customers | Sales, GBP | Sales share |
|---|---:|---:|---:|
{segment_rows}

Scores use tie-preserving percentile quintiles. Recency is measured from midnight
one day after the last observed date. Segment names are heuristic hypotheses.
Recent Low Frequency does not prove a customer is newly acquired.

## Exclusions, applied in order

Each rejected source row gets one reason; counts are mutually exclusive.

| Reason | Rows |
|---|---:|
{excluded}

Repeated rows may be real repeated invoice lines: removal is an explicit assumption
because the source supplies no unique line identifier. Missing-customer sales are
outside this population, so the results do not describe the retailer's entire ledger.

## Interpretation and validation

- December 2011 ends on 9 December. Full-month MoM/YoY comparisons suppress this
  partial month. Annual totals cover different lengths and are not annual growth.
- Customer cohorts use first *observed* purchase. Prior purchases before the source
  window are unknown. Unobserved future cells remain blank; observed inactivity is zero.
- RFM/frequency segments use full history and remain fixed under later report slicers.
- Non-merchandise stock codes remain included; product names use the latest observed
  description. One stock code is one product key, even when descriptions change.
- Country is taken from each invoice line; a customer can appear in several countries.
  Distinct customer/order counts across groups are non-additive.
- Source timezone, profit, ad spend, acquisition channel and costs are unavailable.
- {evidence["checks_passed"]} raw-source/invariant/SQLite comparison groups passed.
  Native Power BI verification is summarized separately in README.md.
  Native MySQL evidence,
  when run, is separate at `data/quality/mysql_validation.json`.

## Figures

![Sales KPIs](figures/kpi_summary.png)
![Monthly sales](figures/monthly_revenue.png)
![Country sales](figures/country_analysis.png)
![Top stock codes](figures/top_products.png)
![Weekday sales](figures/daily_revenue.png)
![Hourly sales](figures/hourly_revenue.png)
![Frequency segments](figures/customer_segments.png)
![RFM](figures/rfm_analysis.png)
![Cohort retention](figures/cohort_retention.png)
"""
    path = ROOT / "power_bi/ANALYSIS.md"
    path.write_text(text, encoding="utf-8")
    print(f"Wrote {path}.")

In [ ]:
write_report(fact, tables, quality, evidence)
print("Read power_bi/ANALYSIS.md, then follow power_bi/README.md to refresh the native dashboard.")

## Explain and hand off

Choose one finding, trace its definition to raw invoice fields, show the independent check and propose a follow-up investigation. Describe the evidence still missing. Optional native MySQL: python sql/run_sql.py --user YOUR_MYSQL_USER. Refresh Power BI only after these checks pass. No profit, causal impact, acquisition or new native validation follows automatically from a Python rerun.